# 01 — POI Observability Diagnostics

This notebook evaluates local-scale observability of daily VIIRS Black Marble VNP46A2 nighttime lights at selected points of interest (POIs) in Samar–Leyte. It uses pre-extracted 5×5 pixel patches and summarises how direct DNB-BRDF observations, Gap-Filled radiance, spatial completeness, retrieval age, and dropout structure vary across POIs and spatial supports.

The objective is to test whether increasing local spatial support from 1×1 to 3×3 and 5×5 pixels improves signal stability, observation availability, or both.

---

In [ ]:
# ==============================
# Code edit: core imports
# ==============================

from pathlib import Path
from ast import literal_eval
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm
import os


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import json

## 1. Input Data and Output Paths

### 1.1 Required input

This notebook uses pre-extracted VNP46A2 patch files for each POI and day. Each `.npz` file contains a 5×5 pixel patch centred on a VIIRS-aligned POI.

| Input                   | Repository path                     | Description                                               |
| ----------------------- | ----------------------------------- | --------------------------------------------------------- |
| 5×5 VNP46A2 POI patches | `datasets/vnp46a2/poi_patches_5x5/` | Daily `.npz` files containing VNP46A2 bands for each POI. |

Expected folder structure:

```text
datasets/
└── vnp46a2/
    └── poi_patches_5x5/
        ├── Tacloban_City/
        │   ├── 2013-01-01.npz
        │   ├── ...
        │   └── 2013-12-31.npz
        ├── Ormoc_City/
        └── ...
```

Each `.npz` patch is expected to include these keys:

| Key                                 | Role                                  |
| ----------------------------------- | ------------------------------------- |
| `DNB_BRDF_Corrected_NTL`            | Directly observed DNB-BRDF radiance.  |
| `Gap_Filled_DNB_BRDF_Corrected_NTL` | Gap-Filled radiance comparator.       |
| `Latest_High_Quality_Retrieval`     | Retrieval-age diagnostic.             |
| `QF_Cloud_Mask`                     | Cloud/retrieval-condition diagnostic. |

In [ ]:
# ==============================
# Project Root and Output Folders
# ==============================

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Main figures: {OUTPUT_MAIN_FIG_DIR}")
print(f"Supplementary figures: {OUTPUT_SUPP_FIG_DIR}")
print(f"Main tables: {OUTPUT_MAIN_TABLE_DIR}")
print(f"Supplementary tables: {OUTPUT_SUPP_TABLE_DIR}")

In [ ]:
def make_plotly_export_safe(obj):
    """
    Convert pandas/numpy/date objects before Plotly export.
    """
    if isinstance(obj, pd.Timestamp):
        return obj.strftime("%Y-%m-%d")

    if isinstance(obj, np.datetime64):
        return pd.Timestamp(obj).strftime("%Y-%m-%d")

    if isinstance(obj, (datetime, date)):
        return obj.isoformat()

    if isinstance(obj, pd.DatetimeIndex):
        return [v.strftime("%Y-%m-%d") for v in obj]

    if isinstance(obj, np.ndarray):
        return [make_plotly_export_safe(v) for v in obj.tolist()]

    if isinstance(obj, list):
        return [make_plotly_export_safe(v) for v in obj]

    if isinstance(obj, tuple):
        return tuple(make_plotly_export_safe(v) for v in obj)

    if isinstance(obj, dict):
        return {k: make_plotly_export_safe(v) for k, v in obj.items()}

    return obj


def export_plotly_figure(
    fig,
    output_path,
    width=1200,
    height=600,
    scale=3,
    export_html=True,
    force_date_axis=False,
    tickformat="%b %d",
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # This converts pandas Timestamp/date objects safely for Kaleido export
    fig_export = go.Figure(json.loads(pio.to_json(fig, validate=False)))

    # Use this only for time-series figures
    if force_date_axis:
        fig_export.update_xaxes(type="date", tickformat=tickformat)

    fig_export.write_image(
        str(output_path.with_suffix(".png")),
        width=width,
        height=height,
        scale=scale,
    )

    fig_export.write_image(
        str(output_path.with_suffix(".svg")),
        width=width,
        height=height,
        scale=scale,
    )

    if export_html:
        fig_export.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs="cdn",
            full_html=True,
        )

    print(f"Exported: {output_path.with_suffix('.png')}")
    print(f"Exported: {output_path.with_suffix('.svg')}")

## 2. Patch Extraction Helpers

### 2.1 Spatial support definitions

Each POI is stored as a 5×5 patch. Three nested spatial supports are extracted from this common patch:

| Extent | Pixels | Description                         |
| ------ | -----: | ----------------------------------- |
| 1×1    |      1 | Centre pixel only.                  |
| 3×3    |      9 | Inner neighbourhood around the POI. |
| 5×5    |     25 | Full local patch.                   |

This design allows the same POIs to be compared across increasing spatial support while preserving a consistent centre location.


In [ ]:
# ==============================
# Code edit: helper functions
# ==============================

def extract_zone(arr, size):
    """
    Extract centre 1×1, inner 3×3, or full 5×5 zone from a 5×5 patch.
    """
    if arr is None:
        return []

    arr = np.array(arr)

    if arr.ndim != 2 or arr.shape != (5, 5):
        return []

    if size == 1:
        return [arr[2, 2]]

    if size == 3:
        return arr[1:4, 1:4].flatten().tolist()

    if size == 5:
        return arr.flatten().tolist()

    return []


def clean_hist(arr):
    """
    Convert patch values to a histogram dictionary.
    """
    return dict(Counter([int(v) for v in arr if v is not None and not pd.isna(v)]))


def summarize_patch(values):
    """
    Compute patch-level radiance summary and spatial completeness.

    In these exported patches, valid DNB-BRDF pixels are represented as
    positive non-missing radiance values.
    """
    arr = np.array([v for v in values if v is not None and not pd.isna(v)], dtype=float)

    valid = arr[arr > 0]

    return {
        "mean": valid.mean() if valid.size else np.nan,
        "std": valid.std() if valid.size else np.nan,
        "valid": int(valid.size),
        "total": int(arr.size),
        "pct": float(valid.size / arr.size * 100) if arr.size else np.nan,
    }


def longest_true_run(arr):
    """
    Return the longest consecutive True run in a Boolean series.
    Used here for maximum dropout duration.
    """
    x = pd.Series(arr).astype(bool).to_numpy()

    if x.size == 0:
        return 0

    padded = np.r_[False, x, False]
    changes = np.diff(padded.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]

    return int((ends - starts).max()) if starts.size else 0

## 3. Aggregate Daily POI Patch Metrics

### 3.1 Build long-format patch table

This section loops through all POI patch folders and extracts daily metrics for the 1×1, 3×3, and 5×5 supports.

The resulting long-format table contains one row per date, POI, and spatial extent.

Expected output columns:

| Column         | Description                                        |
| -------------- | -------------------------------------------------- |
| `date`         | Observation date.                                  |
| `location`     | POI identifier.                                    |
| `extent`       | Spatial support: `1x1`, `3x3`, or `5x5`.           |
| `DNB_mean`     | Mean directly observed DNB-BRDF radiance.          |
| `DNB_stdDev`   | Within-patch DNB-BRDF standard deviation.          |
| `Gap_mean`     | Mean Gap-Filled radiance.                          |
| `Gap_stdDev`   | Within-patch Gap-Filled standard deviation.        |
| `DNB_valid_px` | Number of valid DNB-BRDF pixels.                   |
| `Total_px`     | Number of pixels in the spatial support.           |
| `Valid_pct`    | Spatial completeness percentage.                   |
| `HQ_hist`      | Histogram of latest high-quality retrieval values. |
| `QF_hist`      | Histogram of cloud-mask values.                    |

In [ ]:
patch_dir = "../datasets/vnp46a2/poi_patches_5x5"

In [ ]:
# ==========================================================
# Main Loop — Aggregate All POIs
# ==========================================================

rows = []

for poi in tqdm(os.listdir(patch_dir)):
    
    poi_path = os.path.join(patch_dir, poi)
    
    # Skip non-directory entries (e.g., .DS_Store)
    if not os.path.isdir(poi_path):
        continue

    for filename in os.listdir(poi_path):

        if not filename.endswith(".npz"):
            continue

        date_str = filename.replace(".npz", "")
        path = os.path.join(poi_path, filename)

        try:
            data = np.load(path, allow_pickle=True)
        except:
            continue  # skip corrupted files

        # --------------------------------------------------
        # Loop through spatial extents
        # --------------------------------------------------
        for size, extent in zip([1, 3, 5], ['1x1', '3x3', '5x5']):

            row = {
                'date': date_str,
                'location': poi,
                'extent': extent,
                'DNB_mean': None,
                'DNB_stdDev': None,
                'Gap_mean': None,
                'Gap_stdDev': None,
                'DNB_valid_px': None,
                'Total_px': None,
                'Valid_pct': None,
                'HQ_hist': None,
                'QF_hist': None
            }

            # --------------------------
            # DNB-BRDF
            # --------------------------
            dnb = extract_zone(data.get('DNB_BRDF_Corrected_NTL'), size)
            if dnb:
                stats = summarize_patch(dnb)
                row.update({
                    'DNB_mean': stats['mean'],
                    'DNB_stdDev': stats['std'],
                    'DNB_valid_px': stats['valid'],
                    'Total_px': stats['total'],
                    'Valid_pct': stats['pct']
                })

            # --------------------------
            # Gap-Filled DNB
            # --------------------------
            gap = extract_zone(data.get('Gap_Filled_DNB_BRDF_Corrected_NTL'), size)
            if gap:
                stats = summarize_patch(gap)
                row['Gap_mean'] = stats['mean']
                row['Gap_stdDev'] = stats['std']

            # --------------------------
            # High-Quality Retrieval Histogram
            # --------------------------
            hq = extract_zone(data.get('Latest_High_Quality_Retrieval'), size)
            if hq:
                row['HQ_hist'] = clean_hist(hq)

            # --------------------------
            # Cloud Mask Histogram
            # --------------------------
            qf = extract_zone(data.get('QF_Cloud_Mask'), size)
            if qf:
                row['QF_hist'] = clean_hist(qf)

            rows.append(row)


# ==========================================================
# Convert to DataFrame
# ==========================================================

df = pd.DataFrame(rows)

df.head()

In [ ]:
# ==============================
# Export Supplementary Table: Long POI Patch Metrics
# ==============================

df.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_poi_patch_daily_metrics_long.csv",
    index=False,
)

## 4. Post-Processing and Consistency Checks

### 4.1 Pixel-count consistency

This section standardises pixel denominators across the three spatial supports and recomputes spatial completeness. This ensures that 1×1, 3×3, and 5×5 metrics are comparable.

Spatial completeness is computed as:

$SC = \frac{\text{valid DNB-BRDF pixels}}{\text{total pixels in support}} \times 100$


In [ ]:
# ==========================================================
# Post-processing: Ensure Pixel Count Consistency
# ==========================================================

extent_to_total = {
    "1x1": 1,
    "3x3": 9,
    "5x5": 25,
}

df = df.copy()

df["Total_px"] = df["Total_px"].fillna(df["extent"].map(extent_to_total))
df["DNB_valid_px"] = df["DNB_valid_px"].fillna(0)

df["Valid_pct"] = np.where(
    df["Total_px"] > 0,
    df["DNB_valid_px"] / df["Total_px"] * 100,
    0,
)

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df.head()

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Locations: {df['location'].nunique()}")
print(f"Extents: {sorted(df['extent'].unique())}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

df.groupby(["location", "extent"]).size().head()

## 5. Completeness and Variability Diagnostics

### 5.1 Diagnostic metrics

This section aggregates the daily patch metrics into POI-level diagnostics for each location and spatial extent.

The diagnostic table reports:

| Metric                | Description                                               |
| --------------------- | --------------------------------------------------------- |
| Spatial completeness  | Mean spatial completeness across the time series.         |
| Temporal completeness | Fraction of days satisfying the 50% spatial-support rule. |
| Maximum dropout       | Longest consecutive period failing the 50% rule.          |
| Temporal CV           | Cross-day variability of mean radiance.                   |
| Spatial CV            | Within-patch variability of radiance.                     |

In [ ]:
# ==========================================================
# PREPARATION
# ==========================================================

df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year


# ==========================================================
# 1️⃣ Spatial Completeness (Mean SC per Location × Extent)
# ==========================================================

spatial_completeness = (
    df.groupby(["location", "extent"])["Valid_pct"]
      .mean()
      .reset_index()
      .rename(columns={"Valid_pct": "Spatial_Completeness_mean"})
)


# ==========================================================
# 2️⃣ Temporal Completeness (50% Rule)
# ==========================================================

# Extent-aware 50% thresholds
MIN_VALID_50 = {
    "1x1": 1,
    "3x3": 5,
    "5x5": 13
}

df["min_valid_50"] = df["extent"].map(MIN_VALID_50)
df["meets_50pct"] = df["DNB_valid_px"] >= df["min_valid_50"]


def longest_true_run(arr):
    """
    Return longest consecutive True run in a boolean Series.
    Used to compute maximum dropout duration.
    """
    x = arr.astype(bool).to_numpy()
    if x.size == 0:
        return 0

    padded = np.r_[False, x, False]
    changes = np.diff(padded.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]

    return int((ends - starts).max()) if starts.size else 0


temporal_completeness = (
    df.groupby(["location", "extent"])
      .agg(
          Total_days=("date", "nunique"),
          Valid_days=("meets_50pct", "sum"),
          Max_Dropout_Days=("meets_50pct", lambda s: longest_true_run(~s))
      )
      .reset_index()
)

temporal_completeness["Temporal_Completeness_pct"] = (
    temporal_completeness["Valid_days"] /
    temporal_completeness["Total_days"] * 100
)


# ==========================================================
# 3️⃣ Merge Spatial + Temporal Completeness
# ==========================================================

completeness_summary = pd.merge(
    spatial_completeness,
    temporal_completeness,
    on=["location", "extent"]
)


# ==========================================================
# 4️⃣ Temporal Coefficient of Variation (CV)
# ==========================================================

temporal_cv = (
    df.groupby(["location", "extent", "year"])
      .agg(
          dnb_mean=("DNB_mean", "mean"),
          dnb_std=("DNB_mean", "std"),
          gap_mean=("Gap_mean", "mean"),
          gap_std=("Gap_mean", "std")
      )
      .reset_index()
)

temporal_cv["CV_DNB_temporal"] = (
    temporal_cv["dnb_std"] / temporal_cv["dnb_mean"]
)

temporal_cv["CV_Gap_temporal"] = (
    temporal_cv["gap_std"] / temporal_cv["gap_mean"]
)


# ==========================================================
# 5️⃣ Spatial Coefficient of Variation (CV)
# ==========================================================

df["CV_DNB_spatial"] = df["DNB_stdDev"] / df["DNB_mean"]
df["CV_Gap_spatial"] = df["Gap_stdDev"] / df["Gap_mean"]

spatial_cv = (
    df.groupby(["location", "extent", "year"])
      .agg(
          CV_DNB_spatial=("CV_DNB_spatial", "mean"),
          CV_Gap_spatial=("CV_Gap_spatial", "mean")
      )
      .reset_index()
)


# ==========================================================
# 6️⃣ Merge All Diagnostics
# ==========================================================

cv_summary = pd.merge(
    temporal_cv,
    spatial_cv,
    on=["location", "extent", "year"]
)

diagnostics_summary = (
    completeness_summary
    .merge(
        cv_summary[
            ["location", "extent", "year",
             "CV_DNB_temporal", "CV_Gap_temporal",
             "CV_DNB_spatial", "CV_Gap_spatial"]
        ],
        on=["location", "extent"],
        how="left"
    )
)

diagnostics_summary = diagnostics_summary[
    ["location", "extent", "year",
     "Spatial_Completeness_mean",
     "Valid_days", "Total_days",
     "Temporal_Completeness_pct",
     "Max_Dropout_Days",
     "CV_DNB_temporal", "CV_Gap_temporal",
     "CV_DNB_spatial", "CV_Gap_spatial"]
]

diagnostics_summary

In [ ]:
# ==============================
# Export Supplementary Table: POI Diagnostics Long Table
# ==============================

diagnostics_summary.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_poi_diagnostics_summary_long.csv",
    index=False,
)

## 6. Group-Level Reliability Summary Table

### 6.1 POI groupings

POIs are grouped into three local contexts:

| Group                | POIs                                                                                |
| -------------------- | ----------------------------------------------------------------------------------- |
| City centres         | Tacloban City, Ormoc City, Baybay City                                              |
| Municipal centres    | Palo, Alang-Alang, Guiuan                                                           |
| Non-settlement areas | Samar Forest Reserve, Mt. Nacolod Upland, Central Leyte Forest, Southern Leyte Gulf |

The group-level table reports individual POIs and group averages for spatial completeness, temporal completeness, and maximum dropout duration.


In [ ]:
GROUP_MAP = {
    "Tacloban_City": "City Centers",
    "Ormoc_City": "City Centers",
    "Baybay_City": "City Centers",
    "Palo": "Municipal Centers",
    "Alang_Alang": "Municipal Centers",
    "Guiuan": "Municipal Centers",
    "Samar_Forest_Reserve": "Non-settlement Areas",
    "Mt_Nacolod_Upland": "Non-settlement Areas",
    "Central_Leyte_Forest": "Non-settlement Areas",
    "Southern_Leyte_Gulf": "Non-settlement Areas",
}

RENAME_MAP = {
    "Tacloban_City": "Tacloban City Center",
    "Ormoc_City": "Ormoc City Center",
    "Baybay_City": "Baybay City Center",
    "Alang_Alang": "Alang Alang",
    "Samar_Forest_Reserve": "Samar Forest Reserve",
    "Mt_Nacolod_Upland": "Mt. Nacolod Upland",
    "Central_Leyte_Forest": "Central Leyte Forest",
    "Southern_Leyte_Gulf": "Southern Leyte Gulf",
}

In [ ]:

tbl = diagnostics_summary[
    [
        "location", "extent", "year",
        "Spatial_Completeness_mean",
        "Temporal_Completeness_pct",
        "Max_Dropout_Days",
    ]
].copy()

tbl["group"] = tbl["location"].map(GROUP_MAP)

missing = tbl.loc[tbl["group"].isna(), "location"].unique()
if len(missing) > 0:
    raise RuntimeError(f"Unmapped locations found: {missing}")

group_avg = (
    tbl.groupby(["group", "extent", "year"], observed=False)
    .agg(
        Spatial_Completeness_mean=("Spatial_Completeness_mean", "mean"),
        Temporal_Completeness_pct=("Temporal_Completeness_pct", "mean"),
        Max_Dropout_Days=("Max_Dropout_Days", "mean"),
    )
    .reset_index()
)

group_avg["location"] = group_avg["group"] + " (average)"
group_avg["Max_Dropout_Days"] = group_avg["Max_Dropout_Days"].round().astype(int)

tbl_combined = pd.concat(
    [
        tbl.drop(columns="group"),
        group_avg[
            [
                "location", "extent", "year",
                "Spatial_Completeness_mean",
                "Temporal_Completeness_pct",
                "Max_Dropout_Days",
            ]
        ],
    ],
    ignore_index=True,
)

tbl_combined["location"] = tbl_combined["location"].replace(RENAME_MAP)

row_order = [
    "City Centers (average)",
    "Tacloban City Center",
    "Ormoc City Center",
    "Baybay City Center",
    "Municipal Centers (average)",
    "Palo",
    "Alang Alang",
    "Guiuan",
    "Non-settlement Areas (average)",
    "Samar Forest Reserve",
    "Mt. Nacolod Upland",
    "Central Leyte Forest",
    "Southern Leyte Gulf",
]

tbl_combined["location"] = pd.Categorical(
    tbl_combined["location"],
    categories=row_order,
    ordered=True,
)

tbl_combined = tbl_combined.sort_values(["location", "year", "extent"])

final_table = (
    tbl_combined.pivot_table(
        index=["location", "year"],
        columns="extent",
        values=[
            "Spatial_Completeness_mean",
            "Temporal_Completeness_pct",
            "Max_Dropout_Days",
        ],
        aggfunc="first",
        observed=False,
    )
    .rename(columns={
        "Spatial_Completeness_mean": "Spatial (%)",
        "Temporal_Completeness_pct": "Temporal (%)",
        "Max_Dropout_Days": "Max Dropout Days",
    }, level=0)
    .reindex(["Spatial (%)", "Temporal (%)", "Max Dropout Days"], axis=1, level=0)
    .reindex(["1x1", "3x3", "5x5"], axis=1, level=1)
    .round(1)
)

final_table

In [ ]:
# ==============================
# Export Supplementary Table S2
# Full POI-Level Observability Metrics
# ==============================

final_table.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_s2_full_poi_observability_metrics_2013.csv"
)

In [ ]:
# ==============================
# Export Main Table 2
# POI Kernel Observability Summary
# ==============================

main_table_02 = tbl_combined[
    tbl_combined["location"].astype(str).str.contains("(average)", regex=False)
].copy()

main_table_02["Class"] = (
    main_table_02["location"]
    .astype(str)
    .str.replace(" (average)", "", regex=False)
)

main_table_02["Kernel"] = main_table_02["extent"].map({
    "1x1": "1×1",
    "3x3": "3×3",
    "5x5": "5×5",
})

main_table_02 = main_table_02[
    [
        "Kernel",
        "Class",
        "Spatial_Completeness_mean",
        "Temporal_Completeness_pct",
        "Max_Dropout_Days",
    ]
].copy()

main_table_02 = main_table_02.rename(columns={
    "Spatial_Completeness_mean": "Spatial Completeness SC (%)",
    "Temporal_Completeness_pct": "Temporal Completeness TC (%)",
    "Max_Dropout_Days": "Max Dropout (days)",
})

kernel_order = ["1×1", "3×3", "5×5"]
class_order = ["City Centers", "Municipal Centers", "Non-settlement Areas"]

main_table_02["Kernel"] = pd.Categorical(
    main_table_02["Kernel"],
    categories=kernel_order,
    ordered=True,
)

main_table_02["Class"] = pd.Categorical(
    main_table_02["Class"],
    categories=class_order,
    ordered=True,
)

main_table_02 = (
    main_table_02
    .sort_values(["Kernel", "Class"])
    .round({
        "Spatial Completeness SC (%)": 1,
        "Temporal Completeness TC (%)": 1,
    })
)

main_table_02

In [ ]:
main_table_02.to_csv(
    OUTPUT_MAIN_TABLE_DIR / "table_02_poi_kernel_observability_2013.csv",
    index=False,
)


In [ ]:
ds = diagnostics_summary.copy()
ds["group"] = ds["location"].map(GROUP_MAP)

missing = ds.loc[ds["group"].isna(), "location"].unique()
if len(missing) > 0:
    raise RuntimeError(f"Unmapped locations found: {missing}")

ds["location_disp"] = ds["location"].replace(RENAME_MAP)

long_dnb = ds.melt(
    id_vars=["location", "location_disp", "group", "extent", "year"],
    value_vars=["CV_DNB_spatial", "CV_DNB_temporal"],
    var_name="metric",
    value_name="cv",
)

long_dnb["product"] = "DNB-BRDF"
long_dnb["cv_type"] = long_dnb["metric"].map({
    "CV_DNB_spatial": "Spatial CV",
    "CV_DNB_temporal": "Temporal CV",
})

long_gap = ds.melt(
    id_vars=["location", "location_disp", "group", "extent", "year"],
    value_vars=["CV_Gap_spatial", "CV_Gap_temporal"],
    var_name="metric",
    value_name="cv",
)

long_gap["product"] = "Gap-filled"
long_gap["cv_type"] = long_gap["metric"].map({
    "CV_Gap_spatial": "Spatial CV",
    "CV_Gap_temporal": "Temporal CV",
})

long = pd.concat([long_dnb, long_gap], ignore_index=True)
long = long.dropna(subset=["cv"])

## 7. Variability Diagnostics

### 7.1 Spatial and temporal coefficient of variation

This section compares spatial CV and temporal CV across settlement context and spatial extent. DNB-BRDF variability is shown as boxplots, while Gap-Filled median CV is overlaid as black markers.

This figure distinguishes within-patch heterogeneity from cross-day radiance instability.

In [ ]:
# ======================================================
# ORDERS (SWITCH CV ORDER)
# ======================================================
extent_order = ["1x1", "3x3", "5x5"]
group_order  = ["City Centers", "Municipal Centers", "Non-settlement Areas"]
cv_type_order = ["Spatial CV", "Temporal CV"]  # <-- switched

group_color = {
    "City Centers": "#EF553B",        # red
    "Municipal Centers": "#00CC96",   # green
    "Non-settlement Areas": "#636EFA" # blue
}

long2 = long.copy()
long2["extent"]  = pd.Categorical(long2["extent"], categories=extent_order, ordered=True)
long2["group"]   = pd.Categorical(long2["group"], categories=group_order, ordered=True)
long2["cv_type"] = pd.Categorical(long2["cv_type"], categories=cv_type_order, ordered=True)

# ======================================================
# DATA SPLITS
# ======================================================
dnb = long2[long2["product"] == "DNB-BRDF"]
gap = long2[long2["product"] == "Gap-filled"]

# ======================================================
# BASE BOX PLOT (DNB ONLY)
# ======================================================
fig2 = px.box(
    dnb,
    x="cv",
    y="group",
    color="group",
    facet_row="extent",
    facet_col="cv_type",
    hover_name="location_disp",
    category_orders={
        "extent": extent_order,
        "cv_type": cv_type_order,
        "group": group_order,
    },
    color_discrete_map=group_color,  # <-- FORCE COLORS
    labels={"cv": "Coefficient of Variation", "group": ""},
)

# Clean facet titles
fig2.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("extent=", "").replace("cv_type=", ""),
        font=dict(size=22)
    )
)

fig2.update_traces(
    boxpoints=False,
    line=dict(width=2),
    width=0.6,
)

# ======================================================
# AXIS RANGES (ZERO PRESERVED)
# ======================================================
ranges = {}
pad = 0.01

for cvt in cv_type_order:
    vals = dnb.loc[dnb["cv_type"] == cvt, "cv"].dropna()
    vmin = vals.min()
    vmax = 4
    span = max(vmax - vmin, 1e-6)
    ranges[cvt] = (vmin - pad * span, vmax + pad * span)

# odd xaxes → Spatial CV, even → Temporal CV
for k in fig2.layout:
    if not k.startswith("xaxis"):
        continue
    suffix = k.replace("xaxis", "")
    idx = int(suffix) if suffix else 1
    if idx % 2 == 1:
        fig2.layout[k].update(range=list(ranges["Spatial CV"]))
    else:
        fig2.layout[k].update(range=list(ranges["Temporal CV"]))

# ======================================================
# MEDIANS
# ======================================================

dnb_medians = (
    dnb.groupby(["extent", "cv_type", "group"], observed=True)["cv"]
       .median()
       .reset_index()
)

gap_medians = (
    gap.groupby(["extent", "cv_type", "group"], observed=True)["cv"]
       .median()
       .reset_index()
)

extent_to_row = {ext: len(extent_order) - i for i, ext in enumerate(extent_order)}
cv_to_col     = {cvt: i + 1 for i, cvt in enumerate(cv_type_order)}

# ---- GAP MEDIAN DOTS ----
for i, r in gap_medians.iterrows():
    fig2.add_trace(
        go.Scatter(
            x=[r["cv"]],
            y=[r["group"]],
            mode="markers",
            marker=dict(color="black", size=7),
            name="Gap-filled Median CV" if i == 0 else None,
            showlegend=(i == 0),
        ),
        row=extent_to_row[r["extent"]],
        col=cv_to_col[r["cv_type"]],
    )

# ---- DNB MEDIAN ANNOTATIONS ----
for _, r in dnb_medians.iterrows():
    span = ranges[r["cv_type"]][1] - ranges[r["cv_type"]][0]

    fig2.add_annotation(
        x=r["cv"] + 0.15 * span,
        y=r["group"],
        text=f"{r['cv']:.2f}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=24, color=group_color[r["group"]]),
        row=extent_to_row[r["extent"]],
        col=cv_to_col[r["cv_type"]],
    )

# ======================================================
# PRESENTATION LAYOUT
# ======================================================

fig2.update_layout(
    height=800,
    width=1400,
    margin=dict(l=140, r=260, t=90, b=90),
    paper_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.16,  # aligns with Spatial CV column
        font=dict(size=20),
    ),
    boxmode="group",
)

fig2.update_xaxes(title_font=dict(size=24), tickfont=dict(size=24))
fig2.update_yaxes(tickfont=dict(size=24))



fig_04 = fig2
fig_04.show()

In [ ]:
# ==============================
# Export Main Figure 4
# ==============================

export_plotly_figure(
    fig_04,
    OUTPUT_MAIN_FIG_DIR / "fig_04_poi_cv_by_kernel_and_settlement_2013",
    width=1400,
    height=800,
    scale=3,
)

### 7.2 Plot discussion

Spatial CV increases with larger patch size because broader supports include more within-patch heterogeneity. Temporal CV is generally larger than spatial CV, particularly for non-settlement sites where low baseline radiance makes the signal more sensitive to noise and intermittent retrieval.

The comparison supports the local-scale reliability argument: spatial aggregation can stabilise radiance, but it does not fully resolve temporal instability or observation gaps.



## 8. DNB-BRDF vs Gap-Filled Time Series: Tacloban City

### 8.1 Single-site cross-extent comparison

This figure compares DNB-BRDF and Gap-Filled radiance for Tacloban City across 1×1, 3×3, and 5×5 supports.


In [ ]:
ds = diagnostics_summary.copy()

# assign group FIRST (using raw keys)
ds["group"] = ds["location"].map(GROUP_MAP)

# HARD FAIL — no silent NaNs
missing = ds.loc[ds["group"].isna(), "location"].unique()
if len(missing) > 0:
    raise RuntimeError(f"UNMAPPED LOCATIONS FOUND: {missing}")

# THEN rename for display
ds["location"] = ds["location"].replace(RENAME_MAP)
ds

In [ ]:

site_raw = "Tacloban_City"
site_disp = RENAME_MAP.get(site_raw, site_raw)

subset = df[df["location"] == site_raw].copy()
subset = subset.sort_values("date")

fig_tacloban = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    subplot_titles=("1×1", "3×3", "5×5"),
)

extent_list = ["1x1", "3x3", "5x5"]

for i, extent in enumerate(extent_list, start=1):
    df_e = (
        subset[subset["extent"] == extent]
        .groupby("date", as_index=False)
        .agg({"DNB_mean": "mean", "Gap_mean": "mean"})
        .sort_values("date")
    )

    df_e["date_plot"] = pd.to_datetime(df_e["date"]).dt.to_pydatetime()

    fig_tacloban.add_trace(
        go.Scatter(
            x=df_e["date_plot"],
            y=df_e["Gap_mean"],
            mode="lines",
            line=dict(color="red", width=2),
            name="Gap-Filled",
            showlegend=(i == 1),
        ),
        row=1,
        col=i,
    )

    fig_tacloban.add_trace(
        go.Scatter(
            x=df_e["date_plot"],
            y=df_e["DNB_mean"],
            mode="markers",
            marker=dict(color="blue", size=5),
            name="DNB-BRDF",
            showlegend=(i == 1),
        ),
        row=1,
        col=i,
    )

    fig_tacloban.add_vrect(
        x0=pd.Timestamp("2013-11-06").to_pydatetime(),
        x1=pd.Timestamp("2013-11-13").to_pydatetime(),
        fillcolor="grey",
        opacity=0.3,
        line_width=0,
        row=1,
        col=i,
    )

fig_tacloban.update_layout(
    title=f"DNB-BRDF vs Gap-Filled Time Series — {site_disp}",
    yaxis_title="Radiance (nW·cm⁻²·sr⁻¹)",
    template="simple_white",
    paper_bgcolor="rgba(0,0,0,0)",
    height=400,
    width=1200,
    legend=dict(
        orientation="h",
        y=1.25,
        x=0.8,
        xanchor="center",
        font=dict(size=16),
    ),
)

fig_tacloban.update_xaxes(type="date")
fig_tacloban.show()

In [ ]:
export_plotly_figure(
    fig_tacloban,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_tacloban_dnb_gapfilled_by_kernel_2013",
    width=1200,
    height=400,
    scale=3,
)


### 8.2 Plot discussion

Tacloban shows a strong post-Haiyan radiance decline across all three spatial supports. The single-pixel series is more variable, while 3×3 and 5×5 supports reduce local noise but slightly smooth the magnitude of abrupt changes.

This comparison illustrates the local precision–stability trade-off: broader supports improve visual stability, but the underlying event signal remains dependent on direct observation availability.


## 9. DNB-BRDF vs Gap-Filled Time Series: Representative Sites

### 9.1 Urban, municipal, and non-settlement examples

This figure compares representative POIs across settlement contexts and spatial supports.

In [ ]:
sites_raw = [
    "Tacloban_City",
    "Guiuan",
    "Central_Leyte_Forest",
]

extent_list = ["1x1", "3x3", "5x5"]

fig_rep = make_subplots(
    rows=3,
    cols=3,
    shared_xaxes=True,
    shared_yaxes=True,
    subplot_titles=[
        f"{RENAME_MAP.get(site, site)} — {extent.replace('x', '×')}"
        for site in sites_raw
        for extent in extent_list
    ],
)

for row, site_raw in enumerate(sites_raw, start=1):
    subset = df[df["location"] == site_raw].copy()
    subset = subset.sort_values("date")

    for col, extent in enumerate(extent_list, start=1):
        df_e = (
            subset[subset["extent"] == extent]
            .groupby("date", as_index=False)
            .agg({"DNB_mean": "mean", "Gap_mean": "mean"})
            .sort_values("date")
        )

        df_e["date_plot"] = pd.to_datetime(df_e["date"]).dt.to_pydatetime()

        fig_rep.add_trace(
            go.Scatter(
                x=df_e["date_plot"],
                y=df_e["Gap_mean"],
                mode="lines",
                line=dict(color="red", width=2),
                name="Gap-Filled",
                showlegend=(row == 1 and col == 1),
            ),
            row=row,
            col=col,
        )

        fig_rep.add_trace(
            go.Scatter(
                x=df_e["date_plot"],
                y=df_e["DNB_mean"],
                mode="markers",
                marker=dict(color="blue", size=4),
                name="DNB-BRDF",
                showlegend=(row == 1 and col == 1),
            ),
            row=row,
            col=col,
        )

        fig_rep.add_vrect(
            x0=pd.Timestamp("2013-11-06").to_pydatetime(),
            x1=pd.Timestamp("2013-11-13").to_pydatetime(),
            fillcolor="grey",
            opacity=0.3,
            line_width=0,
            row=row,
            col=col,
        )

fig_rep.update_layout(
    height=600,
    width=1200,
    title="DNB-BRDF vs Gap-Filled Time Series — Representative Sites",
    template="simple_white",
    paper_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        orientation="h",
        y=1.15,
        x=0.8,
        xanchor="center",
        font=dict(size=16),
    ),
)


fig_rep.update_xaxes(type="date")
fig_rep.show()

In [ ]:
export_plotly_figure(
    fig_rep,
    OUTPUT_SUPP_FIG_DIR / "fig_s5_representative_poi_dnb_gapfilled_by_kernel_2013",
    width=1200,
    height=600,
    scale=3,
)

### 9.2 Plot discussion

Urban and municipal POIs show stronger radiance signals and clearer event-related declines than non-settlement areas. Non-settlement sites remain closer to background levels and are more sensitive to noise relative to their mean signal.

The figure reinforces the need for settlement-aware interpretation when using local nighttime-light time series.


## 10. Pre–Post Haiyan DNB-BRDF Differences

### 10.1 Group-level ΔNTL across spatial supports

This figure summarises pre–post Haiyan DNB-BRDF differences for urban, municipal, and non-settlement POI groups across 1×1, 3×3, and 5×5 supports.


In [ ]:
groups = {
    "Urban Core": ["Tacloban_City", "Ormoc_City", "Baybay_City"],
    "Municipal Centers": ["Palo", "Alang_Alang", "Guiuan"],
    "Non-settlement": [
        "Samar_Forest_Reserve",
        "Mt_Nacolod_Upland",
        "Central_Leyte_Forest",
        "Southern_Leyte_Gulf",
    ],
}

extent_list = ["1x1", "3x3", "5x5"]

colors = {
    "Urban Core": "red",
    "Municipal Centers": "green",
    "Non-settlement": "blue",
}

haiyan_date = pd.Timestamp("2013-11-08")

fig_delta = make_subplots(
    rows=3,
    cols=3,
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.05,
    vertical_spacing=0.07,
    subplot_titles=["1×1", "3×3", "5×5"] * 3,
)

for row, (gname, locs) in enumerate(groups.items(), start=1):
    for col, extent in enumerate(extent_list, start=1):
        df_e = df[df["extent"] == extent].copy()
        df_g = df_e[df_e["location"].isin(locs)].copy()

        if df_g.empty:
            continue

        stats = (
            df_g.groupby("date")["DNB_mean"]
            .median()
            .reset_index()
            .sort_values("date")
        )

        stats["date_plot"] = pd.to_datetime(stats["date"]).dt.to_pydatetime()

        pre = df_g[df_g["date"] < haiyan_date]["DNB_mean"]
        post = df_g[df_g["date"] >= haiyan_date]["DNB_mean"]
        delta_mean = post.mean() - pre.mean()

        fig_delta.add_trace(
            go.Scatter(
                x=stats["date_plot"],
                y=stats["DNB_mean"],
                mode="lines",
                line=dict(color=colors[gname], width=2),
                name=gname if col == 1 else None,
                showlegend=(col == 1),
            ),
            row=row,
            col=col,
        )

        fig_delta.add_annotation(
            text=f"ΔNTL = {delta_mean:.2f}",
            xref="x domain",
            yref="y domain",
            x=0.98,
            y=0.9,
            showarrow=False,
            font=dict(size=13, color=colors[gname]),
            row=row,
            col=col,
        )

        fig_delta.add_vrect(
            x0=pd.Timestamp("2013-11-06").to_pydatetime(),
            x1=pd.Timestamp("2013-11-13").to_pydatetime(),
            fillcolor="grey",
            opacity=0.2,
            line_width=0,
            row=row,
            col=col,
        )

        for loc in locs:
            df_loc = df_g[df_g["location"] == loc].sort_values("date").copy()
            df_loc["date_plot"] = pd.to_datetime(df_loc["date"]).dt.to_pydatetime()

            fig_delta.add_trace(
                go.Scatter(
                    x=df_loc["date_plot"],
                    y=df_loc["DNB_mean"],
                    mode="lines",
                    line=dict(color=colors[gname], width=1, dash="dot"),
                    opacity=0.25,
                    showlegend=False,
                ),
                row=row,
                col=col,
            )

fig_delta.update_yaxes(title_text="Urban Core", row=1, col=1)
fig_delta.update_yaxes(title_text="Municipal Centers", row=2, col=1)
fig_delta.update_yaxes(title_text="Non-settlement", row=3, col=1)

fig_delta.update_layout(
    height=600,
    width=1200,
    title="DNB-BRDF ΔNTL Before and After Typhoon Haiyan",
    template="simple_white",
    paper_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=80, r=40, t=80, b=60),
)

fig_delta.add_annotation(
    text="Radiance (nW·cm⁻²·sr⁻¹)",
    x=-0.07,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
)


fig_delta.update_xaxes(type="date")
fig_delta.show()

In [ ]:
export_plotly_figure(
    fig_delta,
    OUTPUT_SUPP_FIG_DIR / "fig_s4_prepost_haiyan_delta_ntl_by_kernel_2013",
    width=1200,
    height=600,
    scale=3,
)

### 10.2 Plot discussion

Urban POIs show the largest negative ΔNTL after Haiyan, municipal centres show smaller but consistent declines, and non-settlement areas remain near background levels. Increasing spatial support dampens local variability but can also smooth the magnitude of abrupt change.

The pattern indicates that event-sensitive radiance changes are settlement-dependent and strongest where anthropogenic lighting is persistent.


## 11. Interactive NTL Patch Time Series Dashboard

### 11.1 Dashboard overview

This dashboard allows rapid inspection of daily spatial completeness and radiance behaviour across POIs and spatial supports.

The upper panel shows daily spatial completeness. The lower panel compares Gap-Filled radiance with directly observed DNB-BRDF radiance, including within-patch variability.


In [ ]:
extent_codes = ["1x1", "3x3", "5x5"]

extent_label_map = {
    "1x1": "1×1 (1 px)",
    "3x3": "3×3 (9 px)",
    "5x5": "5×5 (25 px)",
}

locations = [
    "Tacloban_City", "Ormoc_City", "Baybay_City",
    "Palo", "Alang_Alang", "Guiuan",
    "Samar_Forest_Reserve", "Mt_Nacolod_Upland",
    "Central_Leyte_Forest", "Southern_Leyte_Gulf",
]

fig_dashboard = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.22, 0.78],
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Spatial Completeness (%)",
        "NTL Radiance Time Series",
    ),
)

trace_settings = []

for extent in extent_codes:
    for loc in locations:
        dfl = df[(df["extent"] == extent) & (df["location"] == loc)].copy()

        if dfl.empty:
            continue

        dfl["date_plot"] = pd.to_datetime(dfl["date"]).dt.to_pydatetime()
        dfl = dfl.sort_values("date_plot")

        fig_dashboard.add_trace(
            go.Bar(
                x=dfl["date_plot"],
                y=dfl["Valid_pct"],
                marker_color="forestgreen",
                name="Spatial Completeness",
                showlegend=False,
            ),
            row=1,
            col=1,
        )
        trace_settings.append((extent, loc))

        dfl_gap = dfl.dropna(subset=["Gap_mean", "Gap_stdDev"])

        if not dfl_gap.empty:
            upper = dfl_gap["Gap_mean"] + dfl_gap["Gap_stdDev"]
            lower = dfl_gap["Gap_mean"] - dfl_gap["Gap_stdDev"]

            fig_dashboard.add_trace(
                go.Scatter(
                    x=dfl_gap["date_plot"],
                    y=dfl_gap["Gap_mean"],
                    mode="lines",
                    line=dict(color="red", width=2),
                    name="Gap-Filled",
                    showlegend=True if (extent == "3x3" and loc == locations[0]) else False,
                ),
                row=2,
                col=1,
            )
            trace_settings.append((extent, loc))

            fig_dashboard.add_trace(
                go.Scatter(
                    x=dfl_gap["date_plot"],
                    y=upper,
                    mode="lines",
                    line=dict(width=0),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=2,
                col=1,
            )
            trace_settings.append((extent, loc))

            fig_dashboard.add_trace(
                go.Scatter(
                    x=dfl_gap["date_plot"],
                    y=lower,
                    mode="lines",
                    fill="tonexty",
                    fillcolor="rgba(255,0,0,0.15)",
                    line=dict(width=0),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=2,
                col=1,
            )
            trace_settings.append((extent, loc))

        dfl_dnb = dfl.dropna(subset=["DNB_mean", "DNB_stdDev"])

        if not dfl_dnb.empty:
            fig_dashboard.add_trace(
                go.Scatter(
                    x=dfl_dnb["date_plot"],
                    y=dfl_dnb["DNB_mean"],
                    mode="markers",
                    marker=dict(size=6, color="royalblue"),
                    name="Observed (DNB-BRDF)",
                    error_y=dict(
                        type="data",
                        array=dfl_dnb["DNB_stdDev"],
                        visible=True,
                        thickness=1,
                    ),
                    showlegend=True if (extent == "3x3" and loc == locations[0]) else False,
                ),
                row=2,
                col=1,
            )
            trace_settings.append((extent, loc))


def vis_array(sel_extent, sel_loc):
    return [
        (extent == sel_extent and loc == sel_loc)
        for extent, loc in trace_settings
    ]


location_buttons = [
    dict(
        label=RENAME_MAP.get(loc, loc),
        method="update",
        args=[
            {"visible": vis_array("3x3", loc)},
            {"title": f"NTL Dashboard — 3×3 — {RENAME_MAP.get(loc, loc)}"},
        ],
    )
    for loc in locations
]

extent_buttons = [
    dict(
        label=extent_label_map[extent],
        method="update",
        args=[
            {"visible": vis_array(extent, locations[0])},
            {"title": f"NTL Dashboard — {extent_label_map[extent]} — {RENAME_MAP.get(locations[0], locations[0])}"},
        ],
    )
    for extent in extent_codes
]

initial = vis_array("3x3", locations[0])
for i, vis in enumerate(initial):
    fig_dashboard.data[i].visible = vis

haiyan_date = pd.Timestamp("2013-11-08").to_pydatetime()

fig_dashboard.add_vline(
    x=haiyan_date,
    line_width=2,
    line_dash="dash",
    line_color="black",
)

fig_dashboard.add_annotation(
    x=haiyan_date,
    y=1.05,
    yref="paper",
    text="Haiyan",
    showarrow=False,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    xanchor="left",
)

fig_dashboard.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            buttons=location_buttons,
            x=1.15,
            y=0.65,
            xanchor="left",
        ),
        dict(
            type="buttons",
            buttons=extent_buttons,
            direction="right",
            x=0.98,
            y=1.18,
            xanchor="right",
        ),
    ],
    height=720,
    width=1250,
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        orientation="h",
        y=1.12,
        x=0.5,
        xanchor="center",
    ),
    xaxis2=dict(
        title="Date",
        rangeslider=dict(visible=True),
    ),
    yaxis=dict(
        title="SC (%)",
        range=[0, 100],
    ),
    yaxis2=dict(
        title="Radiance (nW·cm⁻²·sr⁻¹)",
    ),
)


fig_dashboard.update_xaxes(type="date")
fig_dashboard.show()


In [ ]:
fig_dashboard.write_html(
    OUTPUT_SUPP_FIG_DIR / "fig_sx_interactive_poi_dashboard_2013.html",
    include_plotlyjs="cdn",
    full_html=True,
)

### 11.2 Dashboard discussion

The dashboard combines spatial completeness and radiance in a single local-scale view. Drops in spatial completeness indicate days with limited direct observational support, while divergence between DNB-BRDF and Gap-Filled radiance indicates stronger interpolation influence.

The dashboard is retained as an exploratory companion output. Static figures above are used for manuscript-style reporting.


## 12. Summary Tables by Spatial Support

### 12.1 Summary table algorithm

This section generates compact POI summary tables for each spatial support. The tables report valid and gap-filled pixel shares, DNB-BRDF and Gap-Filled radiance statistics, maximum dropout duration, retrieval-age summaries, and same-day high-quality retrieval share.

In [ ]:
from ast import literal_eval

poi_locations = locations

def parse_hist(series):
    merged = {}
    all_vals = []

    for val in series.dropna():
        try:
            h = literal_eval(val) if isinstance(val, str) else val
        except Exception:
            continue

        if not isinstance(h, dict):
            continue

        for k, v in h.items():
            k = int(k)
            v = int(v)
            merged[k] = merged.get(k, 0) + v
            all_vals.extend([k] * v)

    total = sum(merged.values())

    return {
        "rel": {k: (v / total) if total else 0 for k, v in merged.items()},
        "raw": merged,
        "vals": all_vals,
    }


def max_dropout(valid_flags):
    max_run = run = 0

    for v in valid_flags:
        if v == 0:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0

    return max_run

summary_dict = {ec: [] for ec in extent_codes}
extent_to_total = {"1x1": 1, "3x3": 9, "5x5": 25}

for extent_code in extent_codes:
    df_e = df[df["extent"] == extent_code].copy()
    n_pix = extent_to_total[extent_code]

    for loc in poi_locations:
        dfl = df_e[df_e["location"] == loc].copy()

        if dfl.empty:
            continue

        row = {"Location": loc}
        total_days = len(dfl)

        if extent_code == "1x1":
            if "HQ_hist" in dfl.columns:
                hq = parse_hist(dfl["HQ_hist"])
                fresh_days = int(hq["raw"].get(0, 0))
                hq_vals = hq["vals"]
            else:
                fresh_days = dfl["DNB_mean"].notna().sum()
                hq_vals = []

            valid_pct = (fresh_days / total_days * 100) if total_days else 0
            gapfilled_days = total_days - fresh_days

            dnb_mean = dfl["DNB_mean"].mean()
            gap_mean = dfl["Gap_mean"].mean()
            gap_std = dfl["Gap_stdDev"].mean()

            row.update({
                "Valid Pixels (%)": f"{fresh_days} ({valid_pct:.1f}%)",
                "Gap-Filled Pixels (%)": f"{gapfilled_days} ({100 - valid_pct:.1f}%)",
                "DNB Mean": f"{dnb_mean:.2f}" if not np.isnan(dnb_mean) else "—",
                "Gap-Filled Mean ± STD": f"{gap_mean:.2f} ± {gap_std:.2f}" if not np.isnan(gap_mean) else "—",
                "Max Dropout (days)": max_dropout((dfl["DNB_mean"].notna()).astype(int)),
                "Median Latest HQ Retrieval (days)": int(np.median(hq_vals)) if hq_vals else "—",
                "% HQ = 0 (Same-day)": round(valid_pct, 2),
            })

        else:
            total_px = total_days * n_pix
            valid_pix = dfl["DNB_valid_px"].fillna(0).sum()
            gapfilled_pix = total_px - valid_pix

            valid_pct = (valid_pix / total_px * 100) if total_px else 0
            gapfilled_pct = 100 - valid_pct

            dnb_mean = dfl["DNB_mean"].mean()
            dnb_std = dfl["DNB_stdDev"].mean()
            gap_mean = dfl["Gap_mean"].mean()
            gap_std = dfl["Gap_stdDev"].mean()

            if "HQ_hist" in dfl.columns:
                hq = parse_hist(dfl["HQ_hist"])
                hq_vals = hq["vals"]
                median_hq = int(np.median(hq_vals)) if hq_vals else "—"
                pct_hq0 = round(hq["rel"].get(0, 0) * 100, 2)
            else:
                median_hq = "—"
                pct_hq0 = "—"

            row.update({
                "Valid Pixels (%)": f"{int(valid_pix)} ({valid_pct:.1f}%)",
                "Gap-Filled Pixels (%)": f"{int(gapfilled_pix)} ({gapfilled_pct:.1f}%)",
                "DNB Mean ± STD": f"{dnb_mean:.2f} ± {dnb_std:.2f}" if not np.isnan(dnb_mean) else "—",
                "Gap-Filled Mean ± STD": f"{gap_mean:.2f} ± {gap_std:.2f}" if not np.isnan(gap_mean) else "—",
                "Max Dropout (days)": max_dropout((dfl["DNB_valid_px"].fillna(0) > 0).astype(int)),
                "Median Latest HQ Retrieval (days)": median_hq,
                "% HQ = 0 (Same-day)": pct_hq0,
            })

        summary_dict[extent_code].append(row)

summary_1x1 = pd.DataFrame(summary_dict["1x1"])
summary_3x3 = pd.DataFrame(summary_dict["3x3"])
summary_5x5 = pd.DataFrame(summary_dict["5x5"])

In [ ]:
summary_1x1.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_poi_summary_1x1_2013.csv",
    index=False,
)

summary_3x3.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_poi_summary_3x3_2013.csv",
    index=False,
)

summary_5x5.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_poi_summary_5x5_2013.csv",
    index=False,
)

In [ ]:
summary_1x1.head()

In [ ]:
summary_3x3.head()

In [ ]:
summary_5x5.head()

### 12.2 Summary table discussion

The 1×1 table represents single-pixel observability, where valid-pixel counts are equivalent to valid observation days. The 3×3 and 5×5 tables introduce spatial redundancy, increasing the total number of valid pixels but not necessarily eliminating temporal dropout.

Across extents, valid observation percentages remain structurally low, while maximum dropout durations remain broadly similar. This indicates that local spatial aggregation improves radiance stability more than it resolves cloud-driven temporal gaps.


## 13. Notebook Outputs

### Main manuscript outputs

| Output | Path |
|---|---|
| Table 2 — POI kernel observability summary | `outputs/tables/main/table_02_poi_kernel_observability_2013.csv` |
| Figure 4 — POI temporal and spatial CV diagnostics | `outputs/figures/main/fig_04_poi_cv_by_kernel_and_settlement_2013.*` |

### Supplementary outputs

| Output | Path |
|---|---|
| Table S2 — Full POI observability metrics | `outputs/tables/supplementary/table_s2_full_poi_observability_metrics_2013.csv` |
| Long-format POI patch metrics | `outputs/tables/supplementary/table_sx_poi_patch_daily_metrics_long.csv` |
| POI diagnostics summary, long format | `outputs/tables/supplementary/table_sx_poi_diagnostics_summary_long.csv` |
| 1×1 POI summary table | `outputs/tables/supplementary/table_sx_poi_summary_1x1_2013.csv` |
| 3×3 POI summary table | `outputs/tables/supplementary/table_sx_poi_summary_3x3_2013.csv` |
| 5×5 POI summary table | `outputs/tables/supplementary/table_sx_poi_summary_5x5_2013.csv` |
| Figure S4 — Pre–post Haiyan ΔNTL by kernel | `outputs/figures/supplementary/fig_s4_prepost_haiyan_delta_ntl_by_kernel_2013.*` |
| Figure S5 — Representative POI DNB-BRDF vs Gap-Filled by kernel | `outputs/figures/supplementary/fig_s5_representative_poi_dnb_gapfilled_by_kernel_2013.*` |
| Tacloban DNB-BRDF vs Gap-Filled by kernel | `outputs/figures/supplementary/fig_sx_tacloban_dnb_gapfilled_by_kernel_2013.*` |
| Interactive POI dashboard | `outputs/figures/supplementary/fig_sx_interactive_poi_dashboard_2013.html` |

## 14. Summary

This notebook shows that local spatial aggregation improves radiance stability but does not fully resolve temporal observation gaps. Across POIs, valid DNB-BRDF observations remain intermittent, and maximum dropout periods persist across 1×1, 3×3, and 5×5 supports.

The results support the broader reliability framework: local nighttime-light interpretation must account for both spatial support and temporal observability. The next notebook extends this logic from POI-based diagnostics to settlement-stratified aggregation using GHSL SMOD classes.
